# vLLM Slurm Workflow

> Bring up a vLLM OpenAI-compatible endpoint on a Klone/HYAK SLURM compute node and forward it to localhost. The default model is Qwen/Qwen3-8B; the public default account is `stf`, with `amath` available as a local testing override.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#| default_exp vllm

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import test_eq, test_ne, test_fail

In [ ]:
#| export
import argparse
import hashlib
import json
import os
import re
import shlex
import socket
import subprocess
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path

from .core import (
    start_or_connect,
    job_stat,
    update_ssh_node_config,
)


In [ ]:
#| export
DEFAULT_ACCOUNT = "stf"
DEFAULT_PARTITION = "gpu-l40s"
DEFAULT_GRES = "gpu:1"
DEFAULT_CPUS_PER_TASK = 8
DEFAULT_MEM = "48G"
DEFAULT_TIME = "04:00:00"

DEFAULT_MODEL = "Qwen/Qwen3-8B"
DEFAULT_MAX_LEN = 24576

# A pre-built SIF readable by anyone on klone (mode 644). Used as a fallback
# so the first run on a fresh NetID doesn't pay the ~25 min SIF rebuild.
# Override by setting VLLM_SIF in serve.sh's env, or pass shared_sif=None to
# vllm_up to force a per-user build.
SHARED_SIF_PATH = "/mmfs1/gscratch/scrubbed/aurasoph/vllm.sif"


def make_slurm_args(
    *,
    account=DEFAULT_ACCOUNT,
    partition=DEFAULT_PARTITION,
    gres=DEFAULT_GRES,
    cpus_per_task=DEFAULT_CPUS_PER_TASK,
    mem=DEFAULT_MEM,
    time_limit=DEFAULT_TIME,
):
    """Compose the `salloc` resource flags used by `vllm_up`.

    The project default is account=stf. For local testing with amath access,
    call `make_slurm_args(account="amath", time_limit="01:00:00")` or use
    `vllm-up --account amath --time 01:00:00`.
    """
    return (
        f"--account={account} --partition={partition} --gres={gres} "
        f"--cpus-per-task={cpus_per_task} --mem={mem} --time={time_limit}"
    )


DEFAULT_SLURM_ARGS = make_slurm_args()


### `sif_exists` — is the apptainer image on the cluster?

In [ ]:
#| export
def sif_exists(host, sif_path="/mmfs1/gscratch/scrubbed/$USER/vllm.sif"):
    "True iff the apptainer image is on `host`."
    r = subprocess.run(
        ["ssh", "-o", "BatchMode=yes", host, f"[ -f {sif_path} ] && echo yes || echo no"],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        details = "\n".join(
            part.strip()
            for part in (r.stdout, r.stderr)
            if part and part.strip()
        ) or "ssh returned no diagnostic output"
        raise RuntimeError(
            f"ssh failed while checking {sif_path!r} on {host}:\n{details}\n"
            "Run `ssh klone-login true` in your terminal first so HYAK password/Duo "
            "authentication has an active ControlMaster session."
        )
    return r.stdout.strip() == "yes"


### `resolve_sif` — locate a SIF without building

In [ ]:
#| export
def resolve_sif(host, *, shared_sif=SHARED_SIF_PATH):
    """Find a usable SIF on `host` without building.

    Returns the first path that exists on the cluster, or `None` if neither
    is present (in which case the caller should `build_sif`). Order:
      1. /mmfs1/gscratch/scrubbed/$USER/vllm.sif  (the user's own build)
      2. shared_sif (defaults to SHARED_SIF_PATH; a pre-built world-readable copy)
    Pass `shared_sif=None` to skip the fallback.
    """
    user_sif = "/mmfs1/gscratch/scrubbed/$USER/vllm.sif"
    if sif_exists(host, sif_path=user_sif):
        return user_sif
    if shared_sif and sif_exists(host, sif_path=shared_sif):
        return shared_sif
    return None

### `build_sif` — submit the SIF build sbatch and block until done

In [ ]:
#| export
def build_sif(host, remote_vllm_dir="slurm-ops/vllm", silent=False):
    """Submit ~/<remote_vllm_dir>/build-sif.job on `host` and block until done.

    Returns the slurm exit code (0 = sif built). The first build pulls a ~10GB
    docker image and takes ~25-35 min; subsequent calls are no-ops because of
    sif_exists().
    """
    if not silent:
        print(f"[vllm] submitting build-sif.job on {host} (~25-35 min first time)...")
    sub = subprocess.run(
        ["ssh", host, f"cd ~/{remote_vllm_dir} && sbatch --parsable build-sif.job"],
        capture_output=True, text=True, check=True,
    )
    jobid = sub.stdout.strip()
    if not jobid:
        raise RuntimeError(f"sbatch returned no job id; stderr:\n{sub.stderr}")
    if not silent:
        print(f"[vllm] SIF build job: {jobid}; tailing slurm-{jobid}.out (Ctrl-C only stops the tail)")
    tail_script = (
        f"log=~/{remote_vllm_dir}/slurm-{jobid}.out; "
        f"for _ in $(seq 1 600); do [ -f \"$log\" ] && break; sleep 2; done; "
        f"( tail -n +1 -f \"$log\" ) & tp=$!; "
        f"while squeue -h -j {jobid} 2>/dev/null | grep -q .; do sleep 5; done; "
        f"sleep 2; kill $tp 2>/dev/null || true; wait $tp 2>/dev/null || true; "
        f"ec=$(sacct -j {jobid}.batch -o ExitCode -P -n 2>/dev/null | head -1 | cut -d: -f1); "
        f"exit ${{ec:-0}}"
    )
    r = subprocess.run(["ssh", host, tail_script])
    return r.returncode

### `wait_for_discovery` — block until serve.sh publishes the endpoint

In [ ]:
#| export
def wait_for_discovery(job_name, host, timeout_s=1800, poll_s=5, silent=False):
    """Block until ~/.vllm-discovery/<job_name>.json exists on `host`.

    Returns the parsed JSON dict ({\"node\": ..., \"port\": ..., \"model\": ...}).
    Bails early if the SLURM job leaves the queue before the file appears.
    """
    disc_path = f"$HOME/.vllm-discovery/{job_name}.json"
    deadline = time.time() + timeout_s
    if not silent:
        print(f"[vllm] waiting for {disc_path} on {host} (up to {timeout_s//60} min)...")
    while time.time() < deadline:
        r = subprocess.run(
            ["ssh", host, f"cat {disc_path} 2>/dev/null || true"],
            capture_output=True, text=True,
        )
        if r.stdout.strip():
            return json.loads(r.stdout)
        if job_stat(job_name, host, silent=True) is None:
            log = subprocess.run(
                ["ssh", host, f"tail -40 $HOME/.vllm-discovery/{job_name}.log 2>/dev/null"],
                capture_output=True, text=True,
            ).stdout
            raise RuntimeError(
                f"job '{job_name}' is no longer in the queue and no discovery file appeared.\n"
                f"Recent serve log:\n{log}"
            )
        time.sleep(poll_s)
    raise TimeoutError(f"timed out waiting for {disc_path}")

In [ ]:
#| export
def local_port_available(port, host="127.0.0.1"):
    """Return True when `host:port` can be bound locally."""
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.bind((host, int(port)))
    except PermissionError:
        # Some notebook/sandbox environments forbid socket binds. In that case,
        # let ssh be the source of truth instead of failing before Slurm starts.
        return True
    except OSError:
        return False
    return True


def choose_local_port(preferred=8000, *, host="127.0.0.1", attempts=50):
    """Return `preferred` if free, otherwise the next free local port."""
    preferred = int(preferred)
    for port in range(preferred, preferred + attempts + 1):
        if local_port_available(port, host=host):
            return port
    raise RuntimeError(f"no free local port found in [{preferred}, {preferred + attempts}]")


def _forward_target(local_port, node, remote_port):
    return f"{local_port}:{node}.hyak.local:{remote_port}"


def _combined_output(result):
    return "\n".join(
        part.strip()
        for part in (result.stdout, result.stderr)
        if part and part.strip()
    )


def _managed_forward_control_path(job_name, host):
    raw = f"{host}-{job_name}"
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "-", raw).strip("-") or "qwen"
    digest = hashlib.sha1(raw.encode("utf-8")).hexdigest()[:8]
    return Path.home() / ".ssh" / f"slurm-ops-vllm-{safe[:32]}-{digest}.ctl"


def open_ssh_forward(job_name, host, node, remote_port, local_port):
    """Open localhost:`local_port` to `node`:`remote_port` through `host`.

    First tries the user's existing SSH ControlMaster, which avoids another
    password/Duo prompt. If that mux cannot add a forward, fall back to a
    managed tunnel with a dedicated control socket so `vllm_down` can close it.
    """
    target = _forward_target(local_port, node, remote_port)
    mux_cmd = ["ssh", "-O", "forward", "-L", target, host]
    print(f"[vllm] opening forward: {shlex.join(mux_cmd)}")
    mux = subprocess.run(mux_cmd, capture_output=True, text=True)
    if mux.returncode == 0:
        return "controlmaster"

    mux_error = _combined_output(mux)
    if mux_error:
        print(f"[vllm] ControlMaster forward failed:\n{mux_error}")
    print("[vllm] falling back to a managed SSH tunnel")

    control_path = _managed_forward_control_path(job_name, host)
    control_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["ssh", "-S", str(control_path), "-O", "exit", host],
        capture_output=True,
        text=True,
    )
    tunnel_cmd = [
        "ssh",
        "-M",
        "-S",
        str(control_path),
        "-f",
        "-N",
        "-o",
        "ExitOnForwardFailure=yes",
        "-L",
        target,
        host,
    ]
    print(f"[vllm] opening managed tunnel: {shlex.join(tunnel_cmd)}")
    tunnel = subprocess.run(tunnel_cmd, capture_output=True, text=True)
    if tunnel.returncode == 0:
        return f"managed:{control_path}"

    tunnel_error = _combined_output(tunnel)
    details = tunnel_error or mux_error or "ssh returned no diagnostic output"
    raise RuntimeError(
        f"failed to open SSH forward localhost:{local_port} -> {node}.hyak.local:{remote_port}\n"
        f"{details}\n\n"
        "The Slurm job and vLLM server may still be running. Retry with a different local port, for example:\n"
        f"  ./vllm/bin/vllm-up {job_name} {host} --local-port {int(local_port) + 1}\n"
        "Or open the tunnel manually:\n"
        f"  ssh -N -L {target} {host}"
    )


def close_ssh_forward(job_name, host, node, remote_port, local_port):
    """Close either kind of SSH forward opened by `open_ssh_forward`."""
    target = _forward_target(local_port, node, remote_port)
    cancel_cmd = ["ssh", "-O", "cancel", "-L", target, host]
    print(f"[vllm] closing forward: {shlex.join(cancel_cmd)}")
    cancel = subprocess.run(cancel_cmd, capture_output=True, text=True)

    control_path = _managed_forward_control_path(job_name, host)
    exit_cmd = ["ssh", "-S", str(control_path), "-O", "exit", host]
    managed = subprocess.run(exit_cmd, capture_output=True, text=True)

    if cancel.returncode != 0 and managed.returncode != 0:
        print("[vllm] no local SSH forward closed; it may already be gone")


### `vllm_up` - full bring-up

Starts or reconnects to a tmux-backed `salloc`, waits for `serve.sh` to publish the vLLM discovery file, then opens a local SSH tunnel. Tunnel setup is explicit so failures say whether the Slurm job is running and which local port to use.


In [ ]:
#| export
def vllm_up(
    job_name="qwen",
    host="klone-login",
    model=DEFAULT_MODEL,
    slurm_args=DEFAULT_SLURM_ARGS,
    max_len=DEFAULT_MAX_LEN,
    local_port=8000,
    remote_vllm_dir="slurm-ops/vllm",
    update_node_config=True,
    shared_sif=SHARED_SIF_PATH,
):
    """Bring up a vLLM OpenAI-compatible endpoint on Klone and forward it to localhost.

    The public default resources are `make_slurm_args()` (stf / gpu-l40s / gpu:1 / 4h).
    `DEFAULT_MAX_LEN` keeps Qwen3-8B startup predictable for smoke tests.
    For local testing with amath access, pass
    `slurm_args=make_slurm_args(account="amath", time_limit="01:00:00")`.
    """
    requested_local_port = int(local_port)
    local_port = choose_local_port(requested_local_port)
    if local_port != requested_local_port:
        print(f"[vllm] local port {requested_local_port} is busy; using {local_port} instead")

    # 1. Locate (or build) the SIF on the cluster. Prefer the user's own;
    #    fall back to the shared world-readable one to skip the ~25 min build.
    sif_path = resolve_sif(host, shared_sif=shared_sif)
    if sif_path is None:
        print(f"[vllm] no SIF found at $USER's gscratch or shared path; building one")
        rc = build_sif(host, remote_vllm_dir=remote_vllm_dir)
        sif_path = resolve_sif(host, shared_sif=shared_sif)
        if rc != 0 or sif_path is None:
            raise RuntimeError(f"SIF build failed (exit {rc}); see slurm-*.out in ~/{remote_vllm_dir}/")
    print(f"[vllm] using sif: {sif_path}")

    # salloc execs its command directly (no shell), so a shell-style
    # `VLLM_SIF=path srun ...` prefix would fail. srun's --export is the
    # Slurm-native way to inject env into the job. Escape `$` so the laptop
    # shell doesn't expand $USER; Klone expands it when parsing argv for salloc.
    sif_for_shell = sif_path.replace("$", r"\$")
    export_items = [f"VLLM_SIF={sif_for_shell}"]
    if max_len:
        export_items.append(f"MAX_LEN={int(max_len)}")
    serve_cmd = f"srun --pty --export=ALL,{','.join(export_items)} ~/{remote_vllm_dir}/serve.sh"
    full_slurm_args = f"{slurm_args} {serve_cmd}"

    # 2. Reuse upstream's start_or_connect, but run the command it returns.
    ssh_cmd = start_or_connect(job_name, host, slurm_args=full_slurm_args, return_cmd=True)

    if model != DEFAULT_MODEL:
        ssh_cmd = ssh_cmd.replace(
            "tmux new-session -A -s",
            f"MODEL={shlex.quote(model)} tmux new-session -A -s",
            1,
        )

    ssh_cmd_detached = ssh_cmd.replace("tmux new-session -A", "tmux new-session -A -d", 1)
    # Detached tmux does not need a local pseudo-terminal; dropping `ssh -t`
    # makes vllm-up usable from scripts, notebooks, and tests.
    ssh_cmd_detached = ssh_cmd_detached.replace("ssh -t ", "ssh ", 1)
    print(f"[vllm] launching: {ssh_cmd_detached}")
    subprocess.run(ssh_cmd_detached, shell=True, check=True)

    # 3. Wait until salloc allocates a GPU node.
    print("[vllm] waiting for salloc to allocate a GPU node...")
    for _ in range(360):  # 30 min cap on queue wait
        info = job_stat(job_name, host, silent=True)
        if info is not None:
            node, jobid = info
            print(f"[vllm] allocated: job {jobid} on {node}")
            break
        time.sleep(5)
    else:
        raise TimeoutError(f"salloc never reached RUNNING state for '{job_name}'")

    # 4. Wait for vLLM readiness and publish the local forward.
    disc = wait_for_discovery(job_name, host)
    node = disc["node"]
    remote_port = disc["port"]
    forward_method = open_ssh_forward(job_name, host, node, remote_port, local_port)

    if update_node_config:
        try:
            update_ssh_node_config(job_name, host)
        except (RuntimeError, FileNotFoundError) as e:
            print(f"[vllm] (skipped update_ssh_node_config: {e})")

    base_url = f"http://localhost:{local_port}/v1"
    print()
    print("=" * 60)
    print(f"  vLLM ready: {disc['model']} on {node}:{remote_port}")
    print(f"  base_url = {base_url}")
    print(f"  export OPENAI_BASE_URL={base_url!r}")
    print("  export OPENAI_API_KEY='dummy'")
    print(f"  export VLLM_MODEL={disc['served_name']!r}")
    print(f"  try:   ./vllm/bin/vllm-chat --base-url {base_url} 'Reply with exactly: qwen-ready'")
    print(f"  stop:  ./vllm/bin/vllm-down {job_name} {host} --local-port {local_port}")
    print(f"  log:   ssh {host} tail -f ~/.vllm-discovery/{job_name}.log")
    print("=" * 60)
    return {
        "base_url": base_url,
        "model": disc["model"],
        "served_name": disc["served_name"],
        "node": node,
        "local_port": local_port,
        "remote_port": remote_port,
        "forward_method": forward_method,
        "job_name": job_name,
        "host": host,
    }


### `vllm_down` — tear down

In [ ]:
#| export
def vllm_down(job_name="qwen", host="klone-login", local_port=8000, remote_port=None, node=None):
    """Tear down a vllm_up-style session.

    Cancels the Slurm job, kills the tmux session, and cancels the local forward.
    Idempotent.
    """
    info = job_stat(job_name, host, silent=True)
    if (node is None or remote_port is None):
        r = subprocess.run(
            ["ssh", host, f"cat $HOME/.vllm-discovery/{job_name}.json 2>/dev/null || true"],
            capture_output=True, text=True,
        )
        if r.stdout.strip():
            d = json.loads(r.stdout)
            node = node or d.get("node")
            remote_port = remote_port or d.get("port")

    if node and remote_port:
        close_ssh_forward(job_name, host, node, remote_port, local_port)
    else:
        print("[vllm] (no node/remote_port known; skipping local forward cancel)")

    if info is not None:
        _, jobid = info
        print(f"[vllm] scancel {jobid}")
        subprocess.run(["ssh", host, f"scancel {jobid}"])
    else:
        print(f"[vllm] no running '{job_name}' job found; cancelling any pending/completing match")
        subprocess.run(["ssh", host, f"scancel --name={shlex.quote(job_name)} --me 2>/dev/null || true"])

    subprocess.run(["ssh", host, f"tmux kill-session -t {job_name} 2>/dev/null || true"])
    print("[vllm] done.")


### `vllm_chat` - ask the local endpoint a question


In [ ]:
#| export
def _post_json(url, payload, *, api_key="dummy", timeout_s=120):
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(
        url,
        data=data,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {api_key}",
        },
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=timeout_s) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"HTTP {e.code} from {url}: {body}") from e


def _strip_think(text):
    """Remove Qwen3-style <think>...</think> reasoning blocks from a reply."""
    text = re.sub(r"<think>.*?</think>\s*", "", text, flags=re.DOTALL)
    text = re.sub(r"<think>.*", "", text, flags=re.DOTALL)
    return text.strip()


def vllm_chat(
    prompt="Reply with exactly: qwen-ready",
    *,
    base_url=None,
    model=None,
    api_key=None,
    max_tokens=64,
    temperature=0.0,
    timeout_s=120,
    print_response=True,
    no_think=True,
):
    """Send one chat completion request to the local vLLM endpoint.

    Defaults match `vllm_up`: http://localhost:8000/v1, dummy API key,
    and Qwen/Qwen3-8B unless OPENAI_BASE_URL or VLLM_MODEL are set.
    """
    base_url = (base_url or os.environ.get("OPENAI_BASE_URL") or "http://localhost:8000/v1").rstrip("/")
    model = model or os.environ.get("VLLM_MODEL") or DEFAULT_MODEL
    api_key = api_key or os.environ.get("OPENAI_API_KEY") or "dummy"
    if no_think and "/no_think" not in prompt and "/think" not in prompt:
        prompt = prompt.rstrip() + "\n\n/no_think"
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": temperature,
    }
    resp = _post_json(f"{base_url}/chat/completions", payload, api_key=api_key, timeout_s=timeout_s)
    content = resp["choices"][0]["message"]["content"]
    if no_think:
        content = _strip_think(content)
    if print_response:
        print(content)
    return content


### CLI entrypoints used by `vllm/bin/*`


In [ ]:
#| export
def _resource_override_present(args):
    return any(
        getattr(args, name) is not None
        for name in ("account", "partition", "gres", "cpus_per_task", "mem", "time_limit")
    )


def main_up(argv=None):
    ap = argparse.ArgumentParser(prog="vllm-up", description="Start Qwen/vLLM on a Klone Slurm GPU node.")
    ap.add_argument("job_name", nargs="?", default="qwen")
    ap.add_argument("host", nargs="?", default="klone-login")
    ap.add_argument("--model", default=DEFAULT_MODEL)
    ap.add_argument("--max-len", type=int, default=DEFAULT_MAX_LEN,
                    help="vLLM --max-model-len; conservative default for Qwen3-8B smoke tests. Use 0 for model default.")
    ap.add_argument("--local-port", type=int, default=8000, help="Preferred localhost port; if busy, vllm-up uses the next free port")
    ap.add_argument("--account", default=None, help="Slurm account override, e.g. amath for local testing")
    ap.add_argument("--partition", default=None)
    ap.add_argument("--gres", default=None)
    ap.add_argument("--cpus-per-task", dest="cpus_per_task", type=int, default=None)
    ap.add_argument("--mem", default=None)
    ap.add_argument("--time", dest="time_limit", default=None)
    ap.add_argument("--slurm-args", default=None, help="Raw salloc resource string; overrides the resource flags")
    ap.add_argument("--remote-vllm-dir", default="slurm-ops/vllm")
    ap.add_argument("--no-update-node-config", action="store_true")
    args = ap.parse_args(argv)

    slurm_args = args.slurm_args
    if slurm_args is None and _resource_override_present(args):
        slurm_args = make_slurm_args(
            account=args.account or DEFAULT_ACCOUNT,
            partition=args.partition or DEFAULT_PARTITION,
            gres=args.gres or DEFAULT_GRES,
            cpus_per_task=args.cpus_per_task or DEFAULT_CPUS_PER_TASK,
            mem=args.mem or DEFAULT_MEM,
            time_limit=args.time_limit or DEFAULT_TIME,
        )

    kw = {
        "job_name": args.job_name,
        "host": args.host,
        "model": args.model,
        "max_len": args.max_len if args.max_len > 0 else None,
        "local_port": args.local_port,
        "remote_vllm_dir": args.remote_vllm_dir,
        "update_node_config": not args.no_update_node_config,
    }
    if slurm_args is not None:
        kw["slurm_args"] = slurm_args
    try:
        vllm_up(**kw)
    except (RuntimeError, TimeoutError, subprocess.CalledProcessError) as e:
        print(f"vllm-up: {e}", file=sys.stderr)
        return 1
    return 0


def main_down(argv=None):
    ap = argparse.ArgumentParser(prog="vllm-down", description="Stop a vllm-up session.")
    ap.add_argument("job_name", nargs="?", default="qwen")
    ap.add_argument("host", nargs="?", default="klone-login")
    ap.add_argument("--local-port", type=int, default=8000, help="Localhost port printed by vllm-up")
    args = ap.parse_args(argv)
    try:
        vllm_down(args.job_name, args.host, local_port=args.local_port)
    except (RuntimeError, subprocess.CalledProcessError) as e:
        print(f"vllm-down: {e}", file=sys.stderr)
        return 1
    return 0


def main_chat(argv=None):
    ap = argparse.ArgumentParser(prog="vllm-chat", description="Ask the local vLLM endpoint one question.")
    ap.add_argument("prompt", nargs="*", help="Prompt text. Defaults to a qwen-ready smoke test.")
    ap.add_argument("--base-url", default=None)
    ap.add_argument("--model", default=None)
    ap.add_argument("--api-key", default=None)
    ap.add_argument("--max-tokens", type=int, default=64)
    ap.add_argument("--temperature", type=float, default=0.0)
    ap.add_argument("--think", action="store_true", help="Do not append /no_think; allow Qwen3 reasoning output")
    args = ap.parse_args(argv)
    prompt = " ".join(args.prompt).strip() or "Reply with exactly: qwen-ready"
    try:
        vllm_chat(
            prompt,
            base_url=args.base_url,
            model=args.model,
            api_key=args.api_key,
            max_tokens=args.max_tokens,
            temperature=args.temperature,
            no_think=not args.think,
        )
    except (RuntimeError, urllib.error.URLError) as e:
        print(f"vllm-chat: {e}", file=sys.stderr)
        return 1
    return 0


## Usage

```python
from slurm_ops.vllm import make_slurm_args, vllm_up, vllm_chat, vllm_down

# Public default: account=stf, partition=gpu-l40s, gres=gpu:1, time=04:00:00.
info = vllm_up("qwen", "klone-login")
vllm_chat("Reply with exactly: qwen-ready", base_url=info["base_url"], model=info["served_name"])
vllm_down("qwen", "klone-login", local_port=info["local_port"])

# Local testing override for amath access.
info = vllm_up("qwen-test", "klone-login", slurm_args=make_slurm_args(account="amath", time_limit="01:00:00"))
```

Shell equivalents live in `vllm/bin/`:

```bash
./vllm/bin/vllm-up qwen klone-login
./vllm/bin/vllm-chat "Reply with exactly: qwen-ready"
./vllm/bin/vllm-down qwen klone-login

./vllm/bin/vllm-up qwen-test klone-login --account amath --time 01:00:00
```

If `localhost:8000` is already taken, `vllm-up` automatically uses the next free port and prints the exact `base_url`, chat command, and matching `vllm-down --local-port ...` command.


In [ ]:
#| hide
# From the repo root, export only this notebook with:
#   python _scripts/export_vllm.py
